# Stochastic Calculus from Zero · Episode 13: Building the Itô integral

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Three ways to sample · 4. Three answers · 5. Where they come from · 6. The left sum converges · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 13,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

$W_t^3 - 3\int_0^t W_u\,du$ is a martingale: the cube drifts at $3W$ per unit time. Episode 15 derives it from Itô's formula.

In [ ]:
out["prev"] = {"answer": "W^3 - 3 * integral of W"}

## 3. Three ways to sample

Episode 10 showed a Brownian path has infinite length, so $\int W\,dW$ can't be the integral of ordinary calculus. Build it from sums instead: cut $[0, T]$ into steps and add up $W \times \Delta W$, taking $W$ at the **left** end of each step, the **middle** (the average of the two ends), or the **right** end. On a smooth path all three agree in the limit. Here is Episode 1's path cut into 8 steps, with each sampling rule drawn as a step function.

In [ ]:
rng = np.random.default_rng(seed)
t, W, dW = brownian_paths(1, 1_000, 1.0, rng)
w, dw = W[0], dW[0]
assert abs(w[-1] - 0.666204) < 1e-6                      # Episode 1's path
nodes = np.arange(0, 1_001, 125)                          # 8 coarse steps
k = np.minimum(np.arange(1_001) // 125, 7)                # which coarse step each fine point is in
left, right = w[nodes[k]], w[nodes[k + 1]]
every = 2
out["sample"] = {"x": t[::every].round(4).tolist(), "path": w[::every].round(4).tolist(),
                 "left": left[::every].round(4).tolist(), "mid": ((left + right) / 2)[::every].round(4).tolist(),
                 "right": right[::every].round(4).tolist(), "steps": 8,
                 "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}

## 4. Three answers

With all 1,000 steps of Episode 1's path:

In [ ]:
sums = riemann_sums(w, dw)
out["sums"] = sums | {"W_T": float(w[-1]), "half_W2": 0.5 * float(w[-1]) ** 2, "squares": float((dw ** 2).sum()),
                      "chain_ep1": 2 * sums["left"],
                      "table": [{"rule": "Left end", "value": sums["left"]}, {"rule": "Middle", "value": sums["mid"]},
                                {"rule": "Right end", "value": sums["right"]}]}
{k_: v for k_, v in out["sums"].items() if k_ != "table"}

## 5. Where they come from

Three exact identities, true on any path (algebra, not probability):

- **Middle.** $\tfrac12 (W_i + W_{i+1})(W_{i+1} - W_i) = \tfrac12 (W_{i+1}^2 - W_i^2)$ telescopes: the middle sum is exactly $\tfrac12 W_T^2$.
- **Right minus left.** $\sum (W_{i+1} - W_i)\,\Delta W_i = \sum (\Delta W_i)^2$.
- **Left.** So the left sum is $\tfrac12 W_T^2 - \tfrac12 \sum (\Delta W)^2$, and the right sum is $\tfrac12 W_T^2 + \tfrac12 \sum (\Delta W)^2$.

Episode 1's chain-rule sum $\sum 2W\,\Delta W$ was twice the left sum, and its gap from $W_T^2$ was the sum of squared steps. As the steps shrink that sum tends to $T$ (Episode 11), so

$$\int_0^T W\,dW = \tfrac12 W_T^2 - \tfrac12 T \qquad \text{(Itô)}, \qquad \int_0^T W \circ dW = \tfrac12 W_T^2 \qquad \text{(Stratonovich)} .$$

In [ ]:
s = out["sums"]
assert abs(s["mid"] - s["half_W2"]) < 1e-12
assert abs(s["right"] - s["left"] - s["squares"]) < 1e-12
assert abs(s["left"] - (s["half_W2"] - 0.5 * s["squares"])) < 1e-12
assert abs(s["chain_ep1"] + 0.618455) < 1e-5                     # Episode 1's chain-rule sum

## 6. The left sum converges

On Episode 10's fine path, the left sum minus $\tfrac12 W_T^2 - \tfrac12 T$ shrinks to zero as the steps shrink. The limit of left sums is the **Itô integral**.

In [ ]:
_, Wf, _ = brownian_paths(1, 2 ** 20, 1.0, rng)          # Episode 10's fine path (second draw from this seed)
wf = Wf[0]
conv = []
for n in (16, 256, 4_096, 2 ** 20):
    p = wf[:: 2 ** 20 // n]
    conv.append({"steps": n, "gap": riemann_sums(p, np.diff(p))["left"] - (0.5 * p[-1] ** 2 - 0.5)})
assert abs(conv[-1]["gap"]) < 0.01
out["conv"] = {"rows": conv, "fine_gap_abs": abs(conv[-1]["gap"])}
pd.DataFrame(conv)

### Itô or Stratonovich?

Averaged over many paths, the left (Itô) sum is about 0, as a fair game should be (Episode 14), while the middle (Stratonovich) sum averages $E[\tfrac12 W_T^2] = \tfrac12 T$: it keeps the school chain rule but loses the martingale property.

In [ ]:
_, Pm, dPm = brownian_paths(20_000, 500, 1.0, rng)
lefts = (Pm[:, :-1] * dPm).sum(axis=1)
mids = (0.5 * (Pm[:, :-1] + Pm[:, 1:]) * dPm).sum(axis=1)
out["average"] = {"paths": 20_000, "left": float(lefts.mean()), "mid": float(mids.mean()),
                  "table": [{"rule": "Left end (Itô)", "value": float(lefts.mean())},
                            {"rule": "Middle (Stratonovich)", "value": float(mids.mean())}]}
assert abs(out["average"]["left"]) < 0.02 and abs(out["average"]["mid"] - 0.5) < 0.02
out["average"]

## 7. This episode's question

The right-endpoint sum is $\tfrac12 W_T^2 + \tfrac12 \sum(\Delta W)^2 \to \tfrac12 W_T^2 + \tfrac12 T$.

In [ ]:
out["question"] = {"answer": "1/2 W_T^2 + 1/2 T"}

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Show that the middle sum $\sum \tfrac12 (W_i + W_{i+1})(W_{i+1} - W_i)$ equals $\tfrac12 (W_T^2 - W_0^2)$ exactly.

<details><summary>Answer</summary>

Each term is $\tfrac12 (W_{i+1}^2 - W_i^2)$, a difference of squares, and the sum telescopes.

</details>

**Exercise 2 (By hand).** Show that the right sum minus the left sum is $\sum (\Delta W_i)^2$.

<details><summary>Answer</summary>

Term by term: $W_{i+1}\Delta W_i - W_i \Delta W_i = (W_{i+1} - W_i)\Delta W_i = (\Delta W_i)^2$.

</details>

**Exercise 3 (By hand).** What is $\int_0^T 1\,dW$?

<details><summary>Answer</summary>

$W_T - W_0 = W_T$: every sampling rule gives the same telescoping sum.

</details>

**Exercise 4 (In code).** On the fine path, compare left sums for $\int_0^1 W^2\,dW$ with $\tfrac13 W_1^3 - \int_0^1 W\,dt$ (Episode 15 shows why).

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
for n in (256, 4_096, 2 ** 20):
    p = wf[:: 2 ** 20 // n]
    lhs = float((p[:-1] ** 2 * np.diff(p)).sum())
    rhs = p[-1] ** 3 / 3 - float(p[:-1].sum() / n)
    print(f"{n:>8} steps: left sum {lhs:+.4f}, formula {rhs:+.4f}")
assert abs(lhs - rhs) < 0.01

**Exercise 5 (In code).** On 5 new random paths of 2^16 steps, check that the left sum is close to $\tfrac12 W_1^2 - \tfrac12$.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
_, P, dP = brownian_paths(5, 2 ** 16, 1.0, np.random.default_rng(seed + 13))
for p, d in zip(P, dP):
    gap = (p[:-1] * d).sum() - (0.5 * p[-1] ** 2 - 0.5)
    assert abs(gap) < 0.03
print("left sums match 1/2 W^2 - 1/2 T on all 5 paths (to within 0.03)")

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")